In [0]:
%run ./spark_config


In [0]:
# ── Cell 2: Verify ADF landed master file in staging ──────────────
# Ingestion handled by ADF: pl_kaggle_to_staging (daily 5:30 AM IST)
# This cell only confirms file exists before chunking begins

from datetime import datetime

now   = datetime.now()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")

master_partition = (paths["staging"] +
    f"creditcardfraud/year={year}/month={month}/day={day}/")

print("=" * 60)
print(" DATA SIMULATOR — CELL 2: VERIFY STAGING FILE")
print("=" * 60)
print(f"📅 Date      : {year}-{month}-{day}")
print(f"📂 Checking  : {master_partition}")

try:
    files      = dbutils.fs.ls(master_partition)
    data_files = [f for f in files if not f.name.startswith("_")
                                   and not f.name.startswith(".")]

    if len(data_files) == 0:
        raise Exception("No data files found in partition")

    for f in data_files:
        size_mb = round(f.size / (1024*1024), 2)
        print(f"\n✅ Master file confirmed in ADLS staging")
        print(f"   File        : {f.name}")
        print(f"   Size        : {size_mb} MB")
        print(f"   Ingested by : ADF pl_kaggle_to_staging")

    # Store path for Cell 3 and Cell 4
    master_file_path = master_partition + data_files[0].name
    print(f"\n📌 master_file_path → {master_file_path}")

except Exception as e:
    print(f"\n❌ Master file not found in staging")
    print(f"   Expected : {master_partition}")
    print(f"   Fix      : ADF Portal → pl_kaggle_to_staging → Trigger now")
    raise

print("\nCell 2 complete — ready for Cell 3")